**Containers** package your application code, runtime, libraries, and configuration into a single portable unit that runs consistently across any environment. AWS offers managed services to run containers at scale without managing the underlying infrastructure.

The three main container services on AWS are **ECS** (AWS-native orchestration), **EKS** (managed Kubernetes), and **Fargate** (serverless compute engine for both ECS and EKS).

## Containers vs VMs vs Serverless

```
Virtual Machine          Container               Serverless (Lambda)
┌──────────────┐        ┌──────────────┐        ┌──────────────┐
│  App + Libs  │        │  App + Libs  │        │  Function    │
│  Guest OS    │        │  (no OS)     │        │              │
│  Hypervisor  │        │  Container   │        │  Managed     │
│  Host OS     │        │  Runtime     │        │  Runtime     │
│  Hardware    │        │  Host OS     │        │  Hardware    │
└──────────────┘        │  Hardware    │        └──────────────┘
                        └──────────────┘
Slowest to start        Fast (seconds)          Fastest (ms)
Heaviest                Lightweight             No management
Full OS control         Shared OS kernel        15 min max
```

Containers are ideal for: microservices, long-running processes (> 15 min), applications that need more control than Lambda allows, and workloads you want to run identically in dev and prod.

## Amazon ECR — Elastic Container Registry

**ECR** is AWS's managed Docker container image registry — the equivalent of Docker Hub but private and integrated with IAM.

| Feature | Detail |
|---|---|
| **Private registries** | One per AWS account per region; access controlled by IAM |
| **Public registry** | `public.ecr.aws` — share images publicly |
| **Image scanning** | Basic scanning (on push) or Enhanced scanning via Inspector |
| **Lifecycle policies** | Automatically expire old image versions |
| **Replication** | Cross-region and cross-account replication |
| **Encryption** | Images encrypted at rest using KMS |

```bash
# Authenticate Docker to ECR
aws ecr get-login-password | docker login --username AWS \
  --password-stdin 123456789012.dkr.ecr.us-east-1.amazonaws.com

# Tag and push an image
docker tag my-app:latest 123456789012.dkr.ecr.us-east-1.amazonaws.com/my-app:latest
docker push 123456789012.dkr.ecr.us-east-1.amazonaws.com/my-app:latest
```

## Amazon ECS — Elastic Container Service

**ECS** is AWS's native container orchestration service. It manages scheduling, placement, scaling, and health of containers across a cluster.

### ECS building blocks

```
Cluster
  ├── Service A (desired: 3)
  │     ├── Task (running container group)
  │     ├── Task
  │     └── Task
  └── Service B (desired: 2)
        ├── Task
        └── Task
```

| Component | Description |
|---|---|
| **Cluster** | Logical grouping of tasks and services; backed by EC2 or Fargate |
| **Task Definition** | Blueprint — defines containers, CPU/memory, image, ports, volumes, IAM role |
| **Task** | A running instance of a task definition (one or more containers) |
| **Service** | Maintains a desired number of tasks running; integrates with ELB; handles replacements |

## Task Definitions

A task definition is a JSON document that describes how a task (container group) should run:

```json
{
  "family": "web-app",
  "networkMode": "awsvpc",
  "requiresCompatibilities": ["FARGATE"],
  "cpu": "512",
  "memory": "1024",
  "executionRoleArn": "arn:aws:iam::...:role/ecsTaskExecutionRole",
  "taskRoleArn": "arn:aws:iam::...:role/webAppTaskRole",
  "containerDefinitions": [
    {
      "name": "web",
      "image": "123456789012.dkr.ecr.us-east-1.amazonaws.com/my-app:latest",
      "portMappings": [{"containerPort": 80}],
      "logConfiguration": {
        "logDriver": "awslogs",
        "options": {
          "awslogs-group": "/ecs/web-app",
          "awslogs-region": "us-east-1",
          "awslogs-stream-prefix": "web"
        }
      }
    }
  ]
}
```

### Two IAM roles in ECS

| Role | Purpose |
|---|---|
| **Task Execution Role** | Used by ECS agent — pull image from ECR, write logs to CloudWatch |
| **Task Role** | Used by your container code — call S3, DynamoDB, etc. |

## ECS Launch Types: EC2 vs Fargate

| | EC2 Launch Type | Fargate Launch Type |
|---|---|---|
| **Infrastructure** | You manage EC2 instances in the cluster | AWS manages all infrastructure |
| **Visibility** | Full control — choose instance type, AMI | No visibility into underlying host |
| **Cost model** | Pay for EC2 instances (running or idle) | Pay per task vCPU + memory per second |
| **Idle cost** | Yes — instances run even with no tasks | No — zero cost at zero tasks |
| **Maintenance** | Patch, scale, manage EC2 fleet | None |
| **Best for** | Cost optimisation at scale, GPU workloads, special instance requirements | Simplicity, variable workloads, microservices |

### Fargate sizing
Fargate tasks are sized in vCPU and memory combinations:

| vCPU | Memory options |
|---|---|
| 0.25 | 0.5, 1, 2 GB |
| 0.5 | 1–4 GB |
| 1 | 2–8 GB |
| 2 | 4–16 GB |
| 4 | 8–30 GB |
| 8 | 16–60 GB |
| 16 | 32–120 GB |

## ECS Networking

### Network modes

| Mode | How it works | Required for |
|---|---|---|
| **awsvpc** | Each task gets its own ENI and private IP in your VPC | Fargate (mandatory); recommended for EC2 |
| **bridge** | Docker's built-in NAT bridge on the host | EC2 only; legacy |
| **host** | Container shares the host's network namespace | EC2 only; high performance |
| **none** | No external networking | Isolated tasks |

### awsvpc mode (recommended)
- Each task gets its own security group and ENI — full VPC-level control
- Treat each task like a mini EC2 instance from a networking perspective
- Required for Fargate

### ECS + Load Balancer
Attach an ECS Service to an ALB or NLB target group. ECS automatically registers tasks on launch and deregisters on termination. Dynamic port mapping (bridge mode) allows multiple tasks on the same EC2 host to use different host ports.

## ECS Service Auto Scaling

ECS services integrate with **Application Auto Scaling** to adjust the desired task count:

| Policy | Behaviour |
|---|---|
| **Target Tracking** | Keep a metric at a target value (e.g. CPU at 70%, requests per task at 1,000) |
| **Step Scaling** | Add/remove tasks based on CloudWatch alarm thresholds |
| **Scheduled Scaling** | Scale by time of day |

For EC2 launch type, you also need to scale the underlying EC2 capacity — use an **ECS Capacity Provider** backed by an ASG to automate this. Fargate has no underlying capacity to manage.

## Amazon EKS — Elastic Kubernetes Service

**EKS** is a managed Kubernetes service. AWS manages the Kubernetes **control plane** (API server, etcd, scheduler) — you manage the worker nodes (or use Fargate).

### Why Kubernetes / When to choose EKS over ECS

| Reason | Detail |
|---|---|
| **Portability** | Kubernetes is open-source and cloud-agnostic — same YAML configs on AWS, GCP, on-prem |
| **Existing Kubernetes expertise** | Don't retrain teams or rewrite manifests |
| **Rich ecosystem** | Helm charts, operators, service meshes (Istio, Linkerd) |
| **Advanced scheduling** | Node affinity, taints/tolerations, custom schedulers |
| **Large workloads** | Better fit for hundreds of microservices with complex dependencies |

Use **ECS** when you want simplicity and full AWS integration without Kubernetes complexity.  
Use **EKS** when you need Kubernetes-specific features, portability, or are migrating an existing Kubernetes workload.

### EKS architecture

```
EKS Control Plane (AWS managed)
  ├── API Server
  ├── etcd (state store)
  └── Scheduler / Controller Manager
            │
            ▼
Worker Nodes (your VPC)
  ├── Managed Node Group (EC2 ASG, managed by AWS)
  ├── Self-Managed Nodes (you manage EC2 lifecycle)
  └── Fargate Profile (serverless — no nodes to manage)
```

## EKS Node Types

| Node Type | Who manages EC2 | Best for |
|---|---|---|
| **Managed Node Group** | AWS handles launch, update, drain, terminate | Most clusters — simplicity with control |
| **Self-Managed Nodes** | You — using your own AMI and ASG | Custom AMIs, Windows nodes, specific kernel requirements |
| **Fargate Profile** | AWS — no EC2 at all | Burstable microservices, simplicity, no node management |

### EKS Fargate
With a Fargate profile, pods that match the namespace/label selector run on Fargate — AWS allocates compute automatically, no nodes to patch or scale.

Limitations of EKS Fargate:
- No DaemonSets (no per-node agents)
- No privileged containers
- No persistent volumes backed by EBS (use EFS instead)
- No GPU support

## ECS vs EKS — Quick Decision Guide

| Question | ECS | EKS |
|---|---|---|
| New to containers on AWS? | ✓ Simpler | ✗ More complex |
| Already using Kubernetes? | ✗ Different API | ✓ Familiar |
| Need multi-cloud portability? | ✗ AWS-only | ✓ Kubernetes is portable |
| Deep AWS service integration? | ✓ Native | ✓ Via add-ons |
| Serverless containers? | ✓ Fargate | ✓ Fargate (with limits) |
| Advanced scheduling? | ✗ Limited | ✓ Rich Kubernetes scheduler |
| Cost at small scale? | ✓ Cheaper | ✗ EKS control plane $0.10/hr |

> **Exam tip:** For new workloads with no Kubernetes requirement, AWS and exam questions favour ECS + Fargate as the simpler, more cost-effective choice.

## AWS App Runner

**App Runner** is the simplest way to deploy containerised web applications on AWS. You point it at a container image (ECR) or a source code repository, and App Runner handles everything — build, deploy, load balancing, TLS, scaling, and health checks.

```
ECR Image or GitHub Repo
        │
        ▼
    App Runner
  (fully managed)
        │
        ▼
   HTTPS Endpoint
```

| | App Runner | ECS Fargate | Lambda |
|---|---|---|---|
| Setup complexity | Minimal | Medium | Low |
| Max request duration | No limit | No limit | 15 min |
| VPC access | Optional | Yes | Optional |
| Cold starts | Minimal | None (persistent) | Yes |
| Best for | Simple web APIs, small microservices | Production microservices at scale | Event-driven, short functions |

## Working with ECS via boto3

In [ ]:
import boto3
import json

ecs = boto3.client('ecs', region_name='us-east-1')
ecr = boto3.client('ecr', region_name='us-east-1')

In [ ]:
# Create an ECR repository
response = ecr.create_repository(
    repositoryName='my-app',
    imageScanningConfiguration={'scanOnPush': True},
    encryptionConfiguration={'encryptionType': 'KMS'}
)
repo_uri = response['repository']['repositoryUri']
print(f"Repository URI: {repo_uri}")

In [ ]:
# Register a Fargate task definition
response = ecs.register_task_definition(
    family='web-app',
    networkMode='awsvpc',
    requiresCompatibilities=['FARGATE'],
    cpu='512',
    memory='1024',
    executionRoleArn='arn:aws:iam::123456789012:role/ecsTaskExecutionRole',
    taskRoleArn='arn:aws:iam::123456789012:role/webAppTaskRole',
    containerDefinitions=[
        {
            'name': 'web',
            'image': f'{repo_uri}:latest',
            'portMappings': [{'containerPort': 80, 'protocol': 'tcp'}],
            'essential': True,
            'logConfiguration': {
                'logDriver': 'awslogs',
                'options': {
                    'awslogs-group': '/ecs/web-app',
                    'awslogs-region': 'us-east-1',
                    'awslogs-stream-prefix': 'web'
                }
            }
        }
    ]
)
task_def_arn = response['taskDefinition']['taskDefinitionArn']
print(f"Task definition: {task_def_arn}")

In [ ]:
# Create a cluster
ecs.create_cluster(clusterName='demo-cluster')

# Create a Fargate service with ALB integration
response = ecs.create_service(
    cluster='demo-cluster',
    serviceName='web-service',
    taskDefinition='web-app',
    desiredCount=2,
    launchType='FARGATE',
    networkConfiguration={
        'awsvpcConfiguration': {
            'subnets': ['subnet-aaa', 'subnet-bbb'],
            'securityGroups': ['sg-xxxxxxxxx'],
            'assignPublicIp': 'DISABLED'
        }
    },
    loadBalancers=[
        {
            'targetGroupArn': 'arn:aws:elasticloadbalancing:...',  # replace
            'containerName': 'web',
            'containerPort': 80
        }
    ]
)
print(f"Service ARN: {response['service']['serviceArn']}")

In [ ]:
# List running tasks in a service
task_arns = ecs.list_tasks(
    cluster='demo-cluster',
    serviceName='web-service',
    desiredStatus='RUNNING'
)['taskArns']

if task_arns:
    tasks = ecs.describe_tasks(cluster='demo-cluster', tasks=task_arns)['tasks']
    for task in tasks:
        print(f"{task['taskArn'].split('/')[-1]}  {task['lastStatus']}  "
              f"cpu={task['cpu']}  mem={task['memory']}")

In [ ]:
# Deploy a new task definition version (rolling update)
ecs.update_service(
    cluster='demo-cluster',
    service='web-service',
    taskDefinition='web-app:2',    # new revision
    forceNewDeployment=True
)
print("Rolling deployment started")

## Summary

| Concept | Key Takeaway |
|---|---|
| ECR | Private Docker registry; integrated with IAM; scan on push; lifecycle policies |
| ECS Cluster | Logical grouping of tasks/services; backed by EC2 or Fargate |
| Task Definition | Blueprint: image, CPU/memory, ports, log config, IAM roles |
| Task Execution Role | Used by ECS agent to pull images and write logs |
| Task Role | Used by your container code to call AWS services |
| ECS Service | Maintains desired task count; integrates with ALB; handles replacements |
| EC2 launch type | You manage instances; cheaper at scale; needed for GPU/special types |
| Fargate | No infrastructure to manage; pay per task vCPU+memory per second |
| awsvpc network mode | Each task gets its own ENI and security group; required for Fargate |
| EKS | Managed Kubernetes control plane; choose for portability, existing K8s workloads |
| EKS Managed Node Group | AWS handles node lifecycle; recommended default for EKS |
| EKS Fargate | Serverless pods; no nodes; no DaemonSets, no EBS |
| ECS vs EKS | ECS for simplicity + AWS-native; EKS for Kubernetes features + portability |
| App Runner | Simplest container deployment; point at image, get HTTPS endpoint |